# Polarized Tree Demo (`polartox.polarized_tree`)

`PolarizedTree` builds and inspects **one text's** polarized tree directly,
without going through `PolarizedTreesPipeline`. Use this when you already
have one text's annotations and just want to run the splitting algorithm
and look at the result -- no corpus filtering, no cross-text aggregation.

For running the method over a whole corpus and getting F/C/P summaries
across many texts, see `polarized_trees/trees_demo.ipynb`, which uses
`PolarizedTreesPipeline` (built on top of `PolarizedTree`).

The workflow here is:

**generate one text's ratings -> build a PolarizedTree directly -> render
it -> inspect distributions -> query it programmatically**

In [1]:
# Install the package and the nDFU dependency used for polarization scoring.
!pip install polartox
!pip install ndfu

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [2]:
from polartox.datagen import (
    AnnotatorPool,
    DEFAULT_DIMENSIONS,
    DEFAULT_DEPTH_WEIGHTS,
    DEFAULT_INTENSITY_RANGE,
)
from polartox.polarized_tree import PolarizedTree

ModuleNotFoundError: No module named 'polartox.polarized_tree'

## 1. Generate a small dataset and pick one text

`PolarizedTree` works on a single text's rows (one `rating` column plus
whichever demographic dimensions you want to split on). We use the
synthetic generator here only to get a realistic, polarized text to build
a tree from -- `PolarizedTree` itself has no idea the data is synthetic.

In [ ]:
pool = AnnotatorPool(
    dimensions=DEFAULT_DIMENSIONS,
    scale=5,
    intensity_range=DEFAULT_INTENSITY_RANGE,
    depth_weights=DEFAULT_DEPTH_WEIGHTS,
    annotators_per_identity=10,
)

dataset, ground_truth = pool.generate_dataset(
    n_texts=20,
    n_annotators_per_text=None,
    noise=0.05,
    seed=42,
)

# Pick a text with more than one active dimension, so the tree has
# something interesting to find.
text_id = next(
    tid for tid, gt in ground_truth.items() if len(gt["active_dims"]) >= 2
)
text_data = dataset[dataset["text_id"] == text_id]

print(f"Selected text: {text_id}")
print(f"Ground truth active dimensions: {ground_truth[text_id]['active_dims']}")
print(f"Rows for this text: {len(text_data)}")

## 2. Build a PolarizedTree directly

No pipeline object needed -- `PolarizedTree.build(...)` takes the same
splitting parameters (`min_size`, `h`, `max_depth`, `scale`, `theta_pole`,
`theta_stop`, ...) that `detect_polarized_subgroups` and
`PolarizedTreesPipeline` use, and runs the algorithm on just this one
text's rows.

In [ ]:
tree = PolarizedTree.build(
    text_data,
    dims=list(DEFAULT_DIMENSIONS.keys()),
    min_size=10,
    h=0.05,
    max_depth=4,
    scale=pool.scale,
    theta_stop=0.15,
    text_id=text_id,
)

print(f"n_leaves: {tree.n_leaves}")
print(f"depth:    {tree.depth}")

## 3. Render the tree

`render()` prints the same compact ASCII tree used elsewhere in the
package -- one line per node, with each leaf's pole and p_tox.

In [ ]:
tree.render()

## 4. Inspect with rating distributions

`inspect()` walks the whole tree and, with `show_distributions=True`,
prints the rating histogram at every node -- root, internal splits, and
leaves -- so you can see exactly how each split changed the distribution.

In [ ]:
tree.inspect(dataset, show_distributions=True)

## 5. Query the tree programmatically

Beyond printing, `PolarizedTree` exposes the structure directly:

- `get_root()` / `get_leaves()` -- the raw node-dict structures
- `internal_nodes()` -- every non-leaf node, with its split dimension and PRG
- `find_node(path)` -- look up one specific node by its sequence of splits
- `node_distribution(dataset, path)` / `leaf_distributions(dataset)` --
  histograms for one node, or every leaf, without walking the whole tree

In [ ]:
print("Root:", tree.get_root()["is_leaf"], tree.get_root().get("split_dim"))
print()

print("One leaf:")
print(tree.get_leaves()[0])

In [ ]:
for depth, dim, prg, path, node in tree.internal_nodes():
    label = " -> ".join(f"{d}={v}" for d, v in path) or "root"
    print(f"depth={depth}  [{label}]  split on '{dim}'  PRG={prg:.3f}")

In [ ]:
# node_ratings() is the raw building block behind node_distribution() and
# leaf_distributions() -- it just returns the numpy array of ratings for
# one node (root by default), with no printing.
root_ratings = tree.node_ratings(dataset)
print(f"Root ratings: n={len(root_ratings)}, mean={root_ratings.mean():.3f}")

example_leaf = tree.get_leaves()[0]
leaf_ratings = tree.node_ratings(dataset, path=example_leaf["path"])
print(f"Leaf {example_leaf['path']}: n={len(leaf_ratings)}, mean={leaf_ratings.mean():.3f}")

In [ ]:
# Look up one specific leaf's node by its path, and print just that
# node's rating distribution.
example_leaf = tree.get_leaves()[0]
example_path = tuple(example_leaf["path"])

found = tree.find_node(example_path)
print("find_node matches this leaf:", found is example_leaf or found == example_leaf)

tree.node_distribution(dataset, path=example_path)

In [ ]:
# Histogram for every leaf in one call.
tree.leaf_distributions(dataset)

## Where to go next

- **Run this over a whole corpus.** `PolarizedTreesPipeline`
  (`polartox.pipeline`) filters a corpus down to polarized texts, builds a
  `PolarizedTree` per text exactly like this notebook does for one, and
  aggregates F/C/P summaries and diagnostics across all of them --
  see `polarized_trees/trees_demo.ipynb`.
- **Tune the splitting parameters.** `min_size`, `h`, `max_depth`,
  `theta_pole`, and `theta_stop` all shape how deep and how readily this
  tree splits -- see the main package README for what each controls.